In [4]:
!pip install yfinance

  Using cached numpy-2.4.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 14.6 MB/


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import sqlite3
import time
import random
import requests
from datetime import datetime, timedelta
import os

In [2]:
DB_NAME = "sp500_market_data.db"
BENCHMARK_TICKER = "SPY"
DELAY_RANGE = (1.5, 3.0)  # Sleep between 1.5 and 3 seconds to avoid bans

def get_sp500_tickers():
    """Fetches current S&P 500 tickers from Wikipedia."""
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    table = pd.read_html(url)[0]
    tickers = table['Symbol'].tolist()
    # Fix formatting (Wikipedia uses '.' instead of '-' for classes like BRK.B)
    tickers = [t.replace('.', '-') for t in tickers]
    return tickers

In [6]:
tickers = pd.read_csv(r'.\Tickers\SP500.csv')
tickers = tickers['Symbol'].tolist()

In [18]:
stock = tickers_10

In [8]:
tickers_10 = tickers[:10]

In [ ]:
def get_robust_financials(ticker_symbol, hist, fin, bs, cash, info, screening_date):
    """
    Extracts key metrics manually from financial statements to avoid 'None' in .info
    Returns a dictionary of fundamental features.
    """
    features = {}
    
    try:
        # 1. Get Financial Statements (Income Stmt, Balance Sheet)
        fin = ticker_obj.financials
        bs = ticker_obj.balance_sheet
        cash = ticker_obj.cashflow
        
        # Helper to safely get latest value
        def get_latest(df, row_name):
            try:
                if row_name in df.index:
                    return df.loc[row_name].iloc[0] # Most recent year
            except:
                pass
            return np.nan

        # --- EXTRACT METRICS ---
        revenue = get_latest(fin, "Total Revenue")
        net_income = get_latest(fin, "Net Income")
        rnd_expenses = get_latest(fin, "Research And Development")
        
        # Calculate Net Profit Margin
        if revenue and net_income:
            features['Net_Profit_Margin'] = net_income / revenue
        else:
            features['Net_Profit_Margin'] = np.nan
            
        # Calculate R&D Growth (Need at least 2 years)
        try:
            rnd_curr = fin.loc["Research And Development"].iloc[0]
            rnd_prev = fin.loc["Research And Development"].iloc[1]
            features['RnD_Growth'] = (rnd_curr - rnd_prev) / abs(rnd_prev)
        except:
            features['RnD_Growth'] = 0.0 # Assume 0 if no R&D

        # Calculate Sales Growth (3 Year CAGR)
        try:
            sales_curr = fin.loc["Total Revenue"].iloc[0]
            sales_3yr_ago = fin.loc["Total Revenue"].iloc[2] # 3 years ago
            features['Sales_Growth_3yr_CAGR'] = (sales_curr / sales_3yr_ago)**(1/3) - 1
        except:
            features['Sales_Growth_3yr_CAGR'] = np.nan

        # Dividend Yield (Check zero dividend rule)
        div_yield = ticker_obj.info.get('dividendYield', 0)
        features['Dividend_Yield'] = div_yield if div_yield is not None else 0

        # PEG Ratio (Manual Calculation if missing)
        # PEG = (P/E Ratio) / (Earnings Growth Rate)
        pe_ratio = ticker_obj.info.get('trailingPE')
        # We use the calculated sales growth or estimate earnings growth here
        # For simplicity, we store the raw PE to filter later
        features['PE_Ratio'] = pe_ratio
        
        # Capture Current Price for Alpha calculation later
        hist = ticker_obj.history(period="1mo")
        if not hist.empty:
            features['Current_Price'] = hist['Close'].iloc[-1]
            features['Date_Fetched'] = datetime.now().strftime("%Y-%m-%d")
        
    except Exception as e:
        print(f"  Error extracting fundamentals: {e}")
    
    return features

# Rules List

## List 1: Based on Company Data alone 

### Technical Indicators (Price & Volume)
1. Share Price (current) > 50-day moving average (MA)
2. 50 day MA > 200 day MA
3. 13-week RSI > 25-week RSI
4. 20-day On Balance Volume (OBV) is positive

### Valuation & Ratios
1. PEG Ratio > 0.1 and <= 0.5
2. PEG Ratio < 1.0
3. PEG Ratio < 1.5
4. P/B Ratio < 2
5. Dividend Yield = 0% (zero dividend)
6. Market Cap to Cash Flow Ratio < 3

### Financial Health (Balance Sheet/Cash Flow)

1. Debt/Equity (D/E) Ratio < 1
2. Cash Ratio > 1
3. Current Year Free Cash Flow (FCF) > Previous Year
4. Cash Flow from Operations > Net Income
5. Free Cash Flow > 0 
6. ROA (Return on Assets) > 0
7. Cash Ratio of current year > last year
8. Ratio of Cash to Current Liabilities > Last year

## List 2: Based on Company data and industry/market averages 

### Percentile Rankings

1. Market Cap in top 30% (and top 25%)

2. Free Cash Flow in top 30%.

3. PE Ratio in bottom 40% (and lowest 20%).

### Industry comparisons  

#### Valuation

1. PE < Industry Average.

2. P/B < Industry Average.

3. Dividend Yield > Industry Average.

#### Profitability

1. Net Profit Margin > Industry Average.

2. Gross Profit Margin > Industry Average.

3. Operating Margin > Industry Average.

4. ROE > Industry Average.

#### Structure 

1. Ratio of Total Liabilities to Total Assets < Industry Average.

2. Ratio of Long Term Debt to Equity < Industry Average.

3. Ratio of Total Assets to Liabilities > Industry Average.

4. Ratio of Sales to Total Assets > Industry Average.

#### Growth 

1. Sales Growth (3yr) > Industry Average.

2. EPS Growth > Industry Average.

3. Current Year EPS % Change > Industry Average.

4. Net Profit Margin > Industry Avg for 3 years (Requires calculating industry avg for 3 separate years).


## List 3 Historical Data

1. Free Cash Flow growing for 3 years in a row.

2. ROE > 15% for past 3 years.

3. Net Income Growth > 8% (CAGR).

4. Sales Growth > R&D Growth.

5. Operating Margin > Past 3 Year Average (Adapted from 5).


In [10]:
def calculate_rsi(series, period=14):
    '''Calculate RSI using Wilder's smoothing method.'''
    
    delta = series.diff()
    # Gains and Losses 
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    # Wilder's Exponential Weighted Mean (1/period = alpha)
    
    # Subsequent averages: exponential smoothing
    # Wilder's smoothing: alpha = 1/period
    # adjust = False to mathc the formula 
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    
    # RS
    rs = avg_gain / avg_loss
    
    # RSI
    return 100 - (100 / (1 + rs))

def calculate_obv(df):
    '''Function to calculate on Balance Volume (OBV)'''
    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
    return df['OBV']

def get_robust_financials(ticker_obj):
    features = {}
    ticker_symbol = ticker_obj.ticker
    
    try:
        # --- 1. FETCH DATA ---
        hist = ticker_obj.history(period="2y") 
        if hist.empty: return None
        
        fin = ticker_obj.financials
        bs = ticker_obj.balance_sheet
        cash = ticker_obj.cashflow
        info = ticker_obj.info
        
        # --- UNIVERSAL HELPER: Get Single Value Safely ---
        # Replaces both get_item and old get_val
        def get_val(df, keys, iloc_idx=0):
            for k in keys:
                if k in df.index:
                    try:
                        val = df.loc[k].iloc[iloc_idx]
                        return val
                    except IndexError:
                        return np.nan
            return np.nan

        # --- 1a. TECHNICAL INDICATORS ---
        current_price = hist['Close'].iloc[-1]
        
        ma_50 = hist['Close'].rolling(window=50).mean().iloc[-1]
        ma_200 = hist['Close'].rolling(window=200).mean().iloc[-1]
        
        # RSI (Weekly)
        hist_weekly = hist['Close'].resample('W-FRI').last()
        rsi_13_wk = calculate_rsi(hist_weekly, period=13).iloc[-1]
        rsi_25_wk = calculate_rsi(hist_weekly, period=25).iloc[-1]
        
        # OBV
        obv_series = calculate_obv(hist)
        obv_20d_change = obv_series.iloc[-1] - obv_series.iloc[-21] if len(obv_series) > 20 else 0
        
        features['Price_Gt_50MA'] = 1 if current_price > ma_50 else 0
        features['50MA_Gt_200MA'] = 1 if ma_50 > ma_200 else 0
        features['RSI_13W_Gt_RSI_25W'] = 1 if rsi_13_wk > rsi_25_wk else 0
        features['OBV_20D_Positive'] = 1 if obv_20d_change > 0 else 0

        # --- 1b. VALUATION & RATIOS (Calculated) ---
        
        # EPS Data (Using get_val avoids the IndexError crash)
        eps_curr = get_val(fin, ["Basic EPS"], iloc_idx=0)
        eps_prev = get_val(fin, ["Basic EPS"], iloc_idx=1)
        eps_prev_3 = get_val(fin, ["Basic EPS"], iloc_idx=3) # Safely returns NaN if not found
        
        trailing_pe = info.get('trailingPE', np.nan)
        # Fallback (calculate manually
        if pd.isna(trailing_pe):
            # calculate only if EPS positive
            if pd.notna(current_price) and pd.notna(eps_curr) and eps_curr > 0:
                trailing_pe = current_price / eps_curr
            else:
                # If EPS is negative or missing, P/E is undefined (NaN).
                # We do NOT want a negative number here, as it would mess up ranking.
                trailing_pe = np.nan
        
        # Growth Rates
        eps_growth_3yr = np.nan
        
        if pd.notna(eps_curr) and pd.notna(eps_prev_3) and abs(eps_prev_3) > 0:
            
            # CASE A: Standard CAGR (Both Positive)
            # This is the most accurate financial metric for stable companies
            if eps_curr > 0 and eps_prev_3 > 0:
                eps_growth_3yr = (eps_curr / eps_prev_3)**(1/3) - 1
                
            # CASE B: Negative/Mixed Values (Linear Approximation)
            # CAGR formula fails here, so we use: (Total % Change) / 3 Years
            # Formula: ((End - Start) / |Start|) / 3
            else:
                total_change_pct = (eps_curr - eps_prev_3) / abs(eps_prev_3)
                eps_growth_3yr = total_change_pct / 3

        # PEG
        manual_peg = np.nan
        if pd.notna(trailing_pe) and pd.notna(eps_growth_3yr) and eps_growth_3yr > 0:
            manual_peg = trailing_pe / (eps_growth_3yr * 100)
        
        final_peg = manual_peg if pd.notna(manual_peg) else info.get('trailingPegRatio')
        
        features['PEG_01_to_05'] = 1 if (final_peg and 0.1 < final_peg <= 0.5) else 0
        features['PEG_Less_1'] = 1 if (final_peg and final_peg < 1) else 0
        features['PEG_Less_1_5'] = 1 if (final_peg and final_peg < 1.5) else 0 
        
        # Other List 1 Ratios
        pb = info.get('priceToBook')
        features['PB_Less_2'] = 1 if (pb and pb < 2) else 0
        
        div_yield = info.get('dividendYield', 0)
        features['Zero_Dividend'] = 1 if (div_yield is None or div_yield == 0) else 0
        
        mkt_cap = info.get('marketCap', 0)
        ocf_curr = get_val(cash, ["Operating Cash Flow", "Total Cash From Operating Activities"], iloc_idx=0)
        
        if pd.notna(mkt_cap) and pd.notna(ocf_curr) and ocf_curr > 0:
            features['MC_to_CF_Less_3'] = 1 if (mkt_cap / ocf_curr) < 3 else 0
        else:
            features['MC_to_CF_Less_3'] = 0

        # --- 1c. FINANCIAL HEALTH ---
        
        # Fetch Balance Sheet Items
        debt_curr = get_val(bs, ["Total Debt And Capital Lease Obligation", "Total Debt"], iloc_idx=0)
        equity_curr = get_val(bs, ["Total Stockholder Equity", "Stockholders Equity"], iloc_idx=0)
        equity_prev = get_val(bs, ["Total Stockholder Equity", "Stockholders Equity"], iloc_idx=1)
        equity_prev_2 = get_val(bs, ["Total Stockholder Equity", "Stockholders Equity"], iloc_idx=2)
        
        if pd.notna(debt_curr) and pd.notna(equity_curr) and equity_curr != 0:
            features['DE_Less_1'] = 1 if (debt_curr / equity_curr) < 1 else 0
        else:
            features['DE_Less_1'] = 0
            
        # Cash Ratios
        cash_curr = get_val(bs, ["Cash And Cash Equivalents", "Cash Financial"], iloc_idx=0)
        cash_prev = get_val(bs, ["Cash And Cash Equivalents", "Cash Financial"], iloc_idx=1)
        cliab_curr = get_val(bs, ["Current Liabilities", "Total Current Liabilities"], iloc_idx=0)
        cliab_prev = get_val(bs, ["Current Liabilities", "Total Current Liabilities"], iloc_idx=1)
        
        if pd.notna(cash_curr) and pd.notna(cliab_curr) and cliab_curr != 0:
            cash_ratio = cash_curr / cliab_curr
            features['Cash_Ratio_Gt_1'] = 1 if cash_ratio > 1 else 0
            
            if pd.notna(cash_prev) and pd.notna(cliab_prev) and cliab_prev != 0:
                features['Cash_Ratio_Improving'] = 1 if cash_ratio > (cash_prev / cliab_prev) else 0
            else:
                features['Cash_Ratio_Improving'] = 0
        else:
            features['Cash_Ratio_Gt_1'] = 0
            features['Cash_Ratio_Improving'] = 0

        # FCF
        capex_curr = get_val(cash, ["Capital Expenditure", "Capital Expenditures", "Capital Expenditure Reported"], iloc_idx=0)
        capex_prev = get_val(cash, ["Capital Expenditure", "Capital Expenditures", "Capital Expenditure Reported"], iloc_idx=1)
        capex_prev_2 = get_val(cash, ["Capital Expenditure", "Capital Expenditures", "Capital Expenditure Reported"], iloc_idx=2)
        ocf_prev = get_val(cash, ["Operating Cash Flow", "Total Cash From Operating Activities"], iloc_idx=1)
        ocf_prev_2 = get_val(cash, ["Operating Cash Flow", "Total Cash From Operating Activities"], iloc_idx=2)
        
        fcf_curr = (ocf_curr + capex_curr) if (pd.notna(ocf_curr) and pd.notna(capex_curr)) else np.nan
        fcf_prev = (ocf_prev + capex_prev) if (pd.notna(ocf_prev) and pd.notna(capex_prev)) else np.nan
        fcf_prev_2 = (ocf_prev_2 + capex_prev_2) if (pd.notna(ocf_prev_2) and pd.notna(capex_prev_2)) else np.nan
        
        features['FCF_Positive'] = 1 if (pd.notna(fcf_curr) and fcf_curr > 0) else 0
        features['FCF_Growing'] = 1 if (pd.notna(fcf_curr) and pd.notna(fcf_prev) and fcf_curr > fcf_prev) else 0
        
        if pd.notna(fcf_curr) and pd.notna(fcf_prev) and pd.notna(fcf_prev_2):
            fcf_streak = (fcf_curr > fcf_prev) and (fcf_prev > fcf_prev_2)
            features['FCF_Growing_3Yrs_Streak'] = 1 if fcf_streak else 0
        else:
            features['FCF_Growing_3Yrs_Streak'] = 0
        
        # OCF > Net Income
        ni_curr = get_val(fin, ["Net Income", "Net Income Common Stockholders"], iloc_idx=0)
        ni_prev = get_val(fin, ["Net Income", "Net Income Common Stockholders"], iloc_idx=1) 
        ni_prev_2 = get_val(fin, ["Net Income", "Net Income Common Stockholders"], iloc_idx=2)
        ni_prev_3 = get_val(fin, ["Net Income", "Net Income Common Stockholders"], iloc_idx=3)
        features['OCF_Gt_NetIncome'] = 1 if (pd.notna(ocf_curr) and pd.notna(ni_curr) and ocf_curr > ni_curr) else 0
        
        # ROA Positive
        assets_curr = get_val(bs, ["Total Assets"], iloc_idx=0)
        features['ROA_Positive'] = 1 if (pd.notna(ni_curr) and pd.notna(assets_curr) and assets_curr > 0 and ni_curr > 0) else 0
        
        # Fetch Raw Data
        rev_curr = get_val(fin, ['Total Revenue', 'Operating Revenue'], iloc_idx=0)
        rev_prev = get_val(fin, ['Total Revenue', 'Operating Revenue'], iloc_idx=1)
        rev_prev_2 = get_val(fin, ['Total Revenue', 'Operating Revenue'], iloc_idx=2)
        rev_prev_3 = get_val(fin, ['Total Revenue', 'Operating Revenue'], iloc_idx=3)    
        
        # 1d. List 3: Historical Data
        roe_curr = (ni_curr / equity_curr) if (pd.notna(ni_curr) and pd.notna(equity_curr) and equity_curr > 0) else 0
        roe_prev = (ni_prev / equity_prev) if (pd.notna(ni_prev) and pd.notna(equity_prev) and equity_prev > 0) else 0
        roe_prev_2 = (ni_prev_2 / equity_prev_2) if (pd.notna(ni_prev_2) and pd.notna(equity_prev_2) and equity_prev_2 > 0) else 0
        
        features['ROE_Gt_15_3Yrs'] = 1 if (roe_curr > 0.15 and roe_prev > 0.15 and roe_prev_2 > 0.15) else 0
        if pd.notna(ni_curr) and pd.notna(ni_prev_3) and ni_prev_3 > 0 and ni_curr > 0:
            ni_cagr = (ni_curr / ni_prev_3)**(1/3) - 1
            features['Net_Income_Growth_Gt_8pct'] = 1 if ni_cagr > 0.08 else 0
        else:
            features['Net_Income_Growth_Gt_8pct'] = 0  
            
        # Calculate R&D Growth (3 Year CAGR) multiple scenarios to ensure recent R&D spends are considered
        rnd_curr = get_val(fin, ["Research And Development"], iloc_idx=0)
        rnd_prev = get_val(fin, ["Research And Development"], iloc_idx=1)
        rnd_prev_2 = get_val(fin, ["Research And Development"], iloc_idx=2)
        rnd_prev_3 = get_val(fin, ["Research And Development"], iloc_idx=3)
        
        sales_vs_rnd_flag = 0 # Default to 0
        
        # Case 1: Try 3-Year CAGR for BOTH
        if (pd.notna(rev_curr) and pd.notna(rev_prev_3) and rev_prev_3 > 0 and 
            pd.notna(rnd_curr) and pd.notna(rnd_prev_3) and rnd_prev_3 > 0):
            
            s_cagr = (rev_curr / rev_prev_3)**(1/3) - 1
            r_cagr = (rnd_curr / rnd_prev_3)**(1/3) - 1
            sales_vs_rnd_flag = 1 if s_cagr > r_cagr else 0

        # Case 2: Try 2-Year CAGR for BOTH
        elif (pd.notna(rev_curr) and pd.notna(rev_prev_2) and rev_prev_2 > 0 and 
              pd.notna(rnd_curr) and pd.notna(rnd_prev_2) and rnd_prev_2 > 0):
            
            s_cagr = (rev_curr / rev_prev_2)**(1/2) - 1
            r_cagr = (rnd_curr / rnd_prev_2)**(1/2) - 1
            sales_vs_rnd_flag = 1 if s_cagr > r_cagr else 0
            
        # Case 3: Try 1-Year Growth for BOTH
        elif (pd.notna(rev_curr) and pd.notna(rev_prev) and rev_prev > 0 and 
              pd.notna(rnd_curr) and pd.notna(rnd_prev) and rnd_prev > 0):
            
            s_cagr = (rev_curr / rev_prev) - 1
            r_cagr = (rnd_curr / rnd_prev) - 1
            sales_vs_rnd_flag = 1 if s_cagr > r_cagr else 0
            
        # Case 4: Company has Sales Growth but no R&D 
        elif pd.isna(rnd_curr) or rnd_curr == 0:
             # just check for growth
             if pd.notna(rev_curr) and pd.notna(rev_prev) and rev_curr > rev_prev:
                 sales_vs_rnd_flag = 1

        features['Sales_Growth_Gt_RnD_Growth'] = sales_vs_rnd_flag
        
        op_inc_curr = get_val(fin, ['Operating Income'], iloc_idx=0)
        op_inc_prev = get_val(fin, ['Operating Income'], iloc_idx=1)
        op_inc_prev_2 = get_val(fin, ['Operating Income'], iloc_idx=2)
        
        # margins
        om_curr = (op_inc_curr / rev_curr) if (pd.notna(op_inc_curr) and pd.notna(rev_curr)) else 0
        om_prev = (op_inc_prev / rev_prev) if (pd.notna(op_inc_prev) and pd.notna(rev_prev)) else 0
        om_prev_2 = (op_inc_prev_2 / rev_prev_2) if (pd.notna(op_inc_prev_2) and pd.notna(rev_prev_2)) else 0
        
        # average (3 years)
        om_avg_past = (om_curr + om_prev + om_prev_2) / 3
        
        features['Op_Margin_Gt_3yr_Avg'] = 1 if om_curr > om_avg_past else 0
        
        # --- 2. METADATA (Raw Data for List 2) ---
        
        # A. Fetch Raw Data (the ones that are missing)
        
        gross_profit_curr = get_val(fin, ['Gross Profit'], iloc_idx=0)
        
        total_liab_curr = get_val(bs, ['Total Liabilities Net Minority Interest', 'Total Liabilities'], iloc_idx=0)
        if pd.isna(total_liab_curr):
             total_liab_curr = (assets_curr - equity_curr) if (pd.notna(assets_curr) and pd.notna(equity_curr)) else np.nan

        # B. Store Metadata
        features['Ticker'] = ticker_symbol
        features['Sector'] = info.get('sector', 'Unknown')
        features['Market_Cap_Raw'] = mkt_cap 
        features['PE_Raw'] = trailing_pe
        features['Free_Cash_Flow_Raw'] = fcf_curr
        features['PB_Raw'] = pb
        features['Div_Yield_Raw'] = div_yield
        features['ROE_Raw'] = info.get('returnOnEquity')

        # C. Calculate Raw Ratios
        
        # Net Profit Margins (Current, Prev, Prev-2)
        if pd.notna(ni_curr) and pd.notna(rev_curr) and rev_curr > 0:
            features['Net_Profit_Margin_Raw'] = ni_curr / rev_curr
        else:
            features['Net_Profit_Margin_Raw'] = np.nan

        if pd.notna(ni_prev) and pd.notna(rev_prev) and rev_prev > 0:
            features['Net_Profit_Margin_Prev_Raw'] = ni_prev / rev_prev
        else:
            features['Net_Profit_Margin_Prev_Raw'] = np.nan

        if pd.notna(ni_prev_2) and pd.notna(rev_prev_2) and rev_prev_2 > 0:
            features['Net_Profit_Margin_Prev_2_Raw'] = ni_prev_2 / rev_prev_2
        else:
            features['Net_Profit_Margin_Prev_2_Raw'] = np.nan

        # Gross & Operating Margins
        if pd.notna(gross_profit_curr) and pd.notna(rev_curr) and rev_curr > 0:
            features['Gross_Profit_Margin_Raw'] = gross_profit_curr / rev_curr
        else:
            features['Gross_Profit_Margin_Raw'] = np.nan

        if pd.notna(op_inc_curr) and pd.notna(rev_curr) and rev_curr > 0:
            features['Operating_Margin_Raw'] = op_inc_curr / rev_curr
        else:
            features['Operating_Margin_Raw'] = np.nan

        # Sales Growth (3yr)
        if pd.notna(rev_curr) and pd.notna(rev_prev_3) and rev_prev_3 > 0:
             features['Sales_3yr_Growth_Raw'] = (rev_curr / rev_prev_3)**(1/3) - 1
        else:
             features['Sales_3yr_Growth_Raw'] = np.nan

        # Debt & Solvency Ratios
        if pd.notna(total_liab_curr) and pd.notna(assets_curr) and assets_curr > 0:
            features['Debt_To_Assets_Ratio_Raw'] = total_liab_curr / assets_curr
            features['Assets_To_Liability_Ratio_Raw'] = assets_curr / total_liab_curr if total_liab_curr > 0 else np.nan
        else:
            features['Debt_To_Assets_Ratio_Raw'] = np.nan
            features['Assets_To_Liability_Ratio_Raw'] = np.nan

        if pd.notna(debt_curr) and pd.notna(equity_curr) and equity_curr != 0:
            features['Long_Term_Debt_To_Equity_Raw'] = debt_curr / equity_curr
        else:
            features['Long_Term_Debt_To_Equity_Raw'] = np.nan

        if pd.notna(rev_curr) and pd.notna(assets_curr) and assets_curr > 0:
            features['Sales_To_Assets_Ratio_Raw'] = rev_curr / assets_curr
        else:
            features['Sales_To_Assets_Ratio_Raw'] = np.nan

        # EPS Growth Stats
        features['EPS_3yr_Growth_Raw'] = eps_growth_3yr
        
        if pd.notna(eps_curr) and pd.notna(eps_prev) and abs(eps_prev) > 0:
             features['Current_EPS_Change_Raw'] = (eps_curr - eps_prev) / abs(eps_prev)
        else:
             features['Current_EPS_Change_Raw'] = np.nan

        return features
    
    except Exception as e:
        # print(f"Error on {ticker_symbol}: {e}")
        return None

In [11]:
import pandas as pd

In [3]:
df = pd.DataFrame()

In [4]:
df['PZ'] = 254

In [9]:
df = pd.DataFrame({
    'Stock': ['A', 'B', 'C', 'D', 'E'],
    'PE_Raw': [10, 20, 15, 30, 25]
})


In [10]:
df['PE_Percentile'] = df['PE_Raw'].rank(pct=True)

In [12]:
df['PE_Bottom_40_Pct'] = (df['PE_Percentile'] <= 0.40).astype(int)

In [13]:
df

,Stock,PE_Raw,PE_Percentile,PE_Bottom_40_Pct
0,A,10,0.2,1
1,B,20,0.6,0
2,C,15,0.4,1
3,D,30,1.0,0
4,E,25,0.8,0


In [ ]:
def calculate_list_2_rules(df):
    """Calculates Industry Relative and Percentile Rules"""
    print("--- Calculating List 2 (Industry/Universe) Rules ---")

    # --- 1. Percentile Rankings (Universe Wide) ---
    
    # Market Cap
    df['Market_Cap_Percentile'] = df['Market_Cap_Raw'].rank(pct=True)
    df['Market_Cap_Top_30_Pct'] = (df['Market_Cap_Percentile'] >= 0.70).astype(int)
    df['Market_Cap_Top_25_Pct'] = (df['Market_Cap_Percentile'] >= 0.75).astype(int)
    
    # PE Ratio (Lower is better)
    df['PE_Percentile'] = df['PE_Raw'].rank(pct=True)
    df['PE_Bottom_40_Pct'] = (df['PE_Percentile'] <= 0.40).astype(int)
    df['PE_Bottom_20_Pct'] = (df['PE_Percentile'] <= 0.20).astype(int)
    
    # Free Cash Flow
    df['Free_Cash_Flow_Percentile'] = df['Free_Cash_Flow_Raw'].rank(pct=True)
    df['FCF_Top_30_Pct'] = (df['Free_Cash_Flow_Percentile'] >= 0.70).astype(int)

    # --- 2. Industry Relative Rules (Sector Specific) ---
    
    if 'Sector' in df.columns and 'Screening_Date' in df.columns:
        df['Sector_Median_PE'] = df.groupby(['Screening_Date', 'Sector'])['PE_Raw'].transform('median')
        df['PE_Below_Industry'] = (df['PE_Raw'] < df['Sector_Median_PE']).astype(int)
        
        # PE < Industry Median
        df['Sector_Median_PE'] = df.groupby(['Screening_Date', 'Sector'])['PE_Raw'].transform('median')
        df['PE_Below_Industry'] = (df['PE_Raw'] < df['Sector_Median_PE']).astype(int)
        
        # PB < Industry Median
        df['Sector_Median_PB'] = df.groupby(['Screening_Date', 'Sector'])['PB_Raw'].transform('median')
        df['PB_Below_Industry'] = (df['PB_Raw'] < df['Sector_Median_PB']).astype(int)
        
        # Dividend Yield > Industry Median
        df['Sector_Median_Div_Yield'] = df.groupby(['Screening_Date', 'Sector'])['Div_Yield_Raw'].transform('median')
        df['Div_Yield_Above_Industry'] = (df['Div_Yield_Raw'] > df['Sector_Median_Div_Yield']).astype(int)
        
        # Margin > Industry Median
        df['Sector_Median_Margin'] = df.groupby(['Screening_Date', 'Sector'])['Net_Profit_Margin_Raw'].transform('median')
        df['Margin_Above_Industry'] = (df['Net_Profit_Margin_Raw'] > df['Sector_Median_Margin']).astype(int)
        
        # Gross Margin > Industry Median 
        df['Sector_Median_Gross_Margin'] = df.groupby(['Screening_Date', 'Sector'])['Gross_Profit_Margin_Raw'].transform('median')
        df['Gross_Margin_Above_Industry'] = (df['Gross_Profit_Margin_Raw'] > df['Sector_Median_Gross_Margin']).astype(int)      
        
        # Operating Margin > Industry Median
        df['Sector_Median_Operating_Margin'] = df.groupby(['Screening_Date', 'Sector'])['Operating_Margin_Raw'].transform('median')
        df['Operating_Margin_Above_Industry'] = (df['Operating_Margin_Raw'] > df['Sector_Median_Operating_Margin']).astype(int)      
        
        # ROE > Industry Average
        df['Sector_Median_ROE'] = df.groupby(['Screening_Date', 'Sector'])['ROE_Raw'].transform('median')
        df['ROE_Above_Industry'] = (df['ROE_Raw'] > df['Sector_Median_ROE']).astype(int) 
        
        # Ratio of T Liabilities to T Assets < Industry
        df['Sector_Median_Debt_To_Assets_Ratio'] = df.groupby(['Screening_Date', 'Sector'])['Debt_To_Assets_Ratio_Raw'].transform('median')
        df['Debt_To_Assets_Ratio_Below_Industry'] = (df['Debt_To_Assets_Ratio_Raw'] < df['Sector_Median_Debt_To_Assets_Ratio']).astype(int) 
        
        # Ratio of Long Term Debt to Equity < Industry
        df['Sector_Median_Long_Term_Debt_To_Equity'] = df.groupby(['Screening_Date', 'Sector'])['Long_Term_Debt_To_Equity_Raw'].transform('median')
        df['Long_Term_Debt_To_Equity_Ratio_Below_Industry'] = (df['Long_Term_Debt_To_Equity_Raw'] < df['Sector_Median_Long_Term_Debt_To_Equity']).astype(int)         
        
        # Ratio of T Assets to T Liabilities > Industry
        df['Sector_Median_Assets_To_Liability'] = df.groupby(['Screening_Date', 'Sector'])['Assets_To_Liability_Ratio_Raw'].transform('median')
        df['Assets_To_Liability_Ratio_Above_Industry'] = (df['Assets_To_Liability_Ratio_Raw'] > df['Sector_Median_Assets_To_Liability']).astype(int)    
        
        # Ratio of Sales to T Assets > Industry
        df['Sector_Median_Sales_To_Assets'] = df.groupby(['Screening_Date', 'Sector'])['Sales_To_Assets_Ratio_Raw'].transform('median')
        df['Sales_To_Assets_Ratio_Above_Industry'] = (df['Sales_To_Assets_Ratio_Raw'] > df['Sector_Median_Sales_To_Assets']).astype(int)           
        
        # Growth > Industry Median
        df['Sector_Median_Growth'] = df.groupby(['Screening_Date', 'Sector'])['Sales_3yr_Growth_Raw'].transform('median')
        df['Growth_Above_Industry'] = (df['Sales_3yr_Growth_Raw'] > df['Sector_Median_Growth']).astype(int)
        
        # EPS Growth > Industry
        df['Sector_Median_EPS_Growth'] = df.groupby(['Screening_Date', 'Sector'])['EPS_3yr_Growth_Raw'].transform('median')
        df['EPS_Growth_Above_Industry'] = (df['EPS_3yr_Growth_Raw'] > df['Sector_Median_EPS_Growth']).astype(int)       

        # Current Year EPS % Change > Industry
        df['Sector_Median_EPS_Current_Change'] = df.groupby(['Screening_Date', 'Sector'])['Current_EPS_Change_Raw'].transform('median')
        df['EPS_Current_Change_Above_Industry'] = (df['Current_EPS_Change_Raw'] > df['Sector_Median_EPS_Current_Change']).astype(int)       

        # --- 3-Year Margin Streak Rule ---
        # 1. Current Year
        mask_curr_margin = (df['Net_Profit_Margin_Raw'] > df['Sector_Median_Margin'])
        
        # 2. Previous Year
        df['Sector_Median_Margin_Prev'] = df.groupby(['Screening_Date', 'Sector'])['Net_Profit_Margin_Prev_Raw'].transform('median')
        mask_prev_margin = (df['Net_Profit_Margin_Prev_Raw'] > df['Sector_Median_Margin_Prev'])  
        
        # 3. Two Years Ago (FIX: Changed Prev_3 to Prev_2 to match metadata)
        df['Sector_Median_Margin_Prev_2_yr'] = df.groupby(['Screening_Date', 'Sector'])['Net_Profit_Margin_Prev_2_Raw'].transform('median')
        mask_prev_2_margin = (df['Net_Profit_Margin_Prev_2_Raw'] > df['Sector_Median_Margin_Prev_2_yr'])  

        # Combine: Must win all 3 years
        df['Margin_Gt_Industry_3yr_Streak'] = (mask_curr_margin & mask_prev_margin & mask_prev_2_margin).astype(int)
        
        # --- ROE e year streak rule 
        
        
    # 3. Cleanup 
    # List of all temporary median/percentile columns to drop
    cols_to_drop = [
        'Market_Cap_Percentile', 'PE_Percentile', 'Free_Cash_Flow_Percentile',
        'Sector_Median_PE', 'Sector_Median_PB', 'Sector_Median_Div_Yield',
        'Sector_Median_Margin', 'Sector_Median_Gross_Margin', 'Sector_Median_Operating_Margin',
        'Sector_Median_ROE', 'Sector_Median_Debt_To_Assets_Ratio', 
        'Sector_Median_Long_Term_Debt_To_Equity', 'Sector_Median_Assets_To_Liability',
        'Sector_Median_Sales_To_Assets', 'Sector_Median_Growth', 'Sector_Median_EPS_Growth',
        'Sector_Median_EPS_Current_Change', 'Sector_Median_Margin_Prev', 'Sector_Median_Margin_Prev_2_yr'
    ]
    df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    
    print("Success: List 2 Rules Added.")
    return df

In [13]:
import os
import sqlite3
import pandas as pd
import time
import random
import yfinance as yf

# --- CONFIGURATION ---
DB_NAME = "market_data_universe.db"  # Renamed to reflect broader scope
OUTPUT_CSV = "final_strategy_results.csv"
DELAY_RANGE = (1.0, 2.0)

# --- THE SWITCH ---
# Set to FALSE to just collect data (fast)
# Set to TRUE to process the industry stats (only do this after collecting everyone)
CALCULATE_LIST_2 = False 

def main():
    print(f"=== PHASE 1: Data Collection (Mode: {'Analysis' if CALCULATE_LIST_2 else 'Collection Only'}) ===")
    
    conn = sqlite3.connect(DB_NAME)
    
    # 1. Load Tickers
    # Ideally, replace this with a larger list (e.g., NASDAQ-3000 + NYSE)
    tickers = tickers_10 
    
    # PRO TIP: Shuffle the list. 
    # If the script crashes or you stop it, you won't be stuck only having "A" stocks.
    random.shuffle(tickers) 
    
    successful_counts = 0
    
    for ticker in tickers:
        print(f"Processing {ticker}...", end=" ")
        
        try:
            # --- DUPLICATE CHECK (Super fast SQL query) ---
            try:
                # Check if ticker exists
                cursor = conn.cursor()
                cursor.execute("SELECT 1 FROM fundamentals WHERE Ticker = ?", (ticker,))
                if cursor.fetchone():
                    print("Skipping (Already in DB)")
                    continue
            except sqlite3.OperationalError:
                # Table doesn't exist yet, safe to proceed
                pass

            # --- FETCH DATA ---
            stock = yf.Ticker(ticker)
            fund_data = get_robust_financials(stock)
            
            if not fund_data: 
                print("Skipping (No Data)")
                continue
            
            # Count how many values in the dictionary are NaN
            missing_count = sum(1 for v in fund_data.values() if pd.isna(v))
            total_fields = len(fund_data)
            
            # Print status based on quality
            if missing_count == 0:
                status_msg = "Perfect Data"
            elif missing_count < 5:
                status_msg = f"Good ({missing_count} missing)"
            else:
                status_msg = f"WARNING: Gappy Data ({missing_count}/{total_fields} missing)"
            
            # Save to DB
            df_row = pd.DataFrame([fund_data])
            df_row.to_sql('fundamentals', conn, if_exists='append', index=False)
            
            # Update print to show the status
            print(f"Done. [{status_msg}]")
            
            # --- SAVE TO DB ---
            df_row = pd.DataFrame([fund_data])
            df_row.to_sql('fundamentals', conn, if_exists='append', index=False)
            
            print("Done.")
            successful_counts += 1
            
            # Sleep to avoid IP ban
            time.sleep(random.uniform(*DELAY_RANGE))
            
        except Exception as e:
            print(f"Failed: {e}")

    print(f"\nPhase 1 Complete. Added {successful_counts} NEW stocks.")
    
    # --- PHASE 2: Industry Analysis (Conditional) ---
    if CALCULATE_LIST_2:
        print("\n=== PHASE 2: Calculating Context (List 2) ===")
        try:
            print(f"Loading UNIVERSE from {DB_NAME}...")
            # Load the entire massive dataset
            df_universe = pd.read_sql("SELECT * FROM fundamentals", conn)
            conn.close() 
            
            if not df_universe.empty:
                print(f"Dataset Size: {len(df_universe)} stocks.")
                
                # Apply the relative rules against this broader universe
                df_final = calculate_list_2_rules(df_universe)
                
                print(f"Saving combined dataset to {OUTPUT_CSV}...")
                df_final.to_csv(OUTPUT_CSV, index=False)
                print("Success! Analysis Complete.")
            else:
                print("Database is empty.")
        except Exception as e:
            print(f"Error in Phase 2: {e}")
    else:
        print("\nSkipping Phase 2 (Collection Mode).")
        print(f"Data is safely stored in {DB_NAME}. Change CALCULATE_LIST_2 = True to generate CSV.")
        conn.close()

if __name__ == "__main__":
    main()

=== PHASE 1: Data Collection (Mode: Collection Only) ===
Processing ACN... Skipping (Already in DB)
Processing AYI... Skipping (Already in DB)
Processing AMD... Skipping (Already in DB)
Processing AAP... Skipping (Already in DB)
Processing ATVI... 

$ATVI: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


Skipping (No Data)
Processing ADBE... Skipping (Already in DB)
Processing MMM... Skipping (Already in DB)
Processing AOS... Skipping (Already in DB)
Processing ABT... Skipping (Already in DB)
Processing ABBV... Skipping (Already in DB)

Phase 1 Complete. Added 0 NEW stocks.

Skipping Phase 2 (Collection Mode).
Data is safely stored in market_data_universe.db. Change CALCULATE_LIST_2 = True to generate CSV.


In [50]:
import pandas as pd
import sqlite3

DB_NAME = "market_data_universe.db" # Or "sp500_market_data.db"
OUTPUT_AUDIT_CSV = "detailed_missing_data_report.csv"

def get_missing_columns(row):
    """Returns a comma-separated string of column names that are NaN for this row."""
    # row.index is the column name, row.values is the value
    # We filter for indices where the value is null
    missing_cols = row.index[row.isna()].tolist()
    return ", ".join(missing_cols)

def main():
    print(f"Reading data from {DB_NAME}...")
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql("SELECT * FROM fundamentals", conn)
    conn.close()

    print(f"Auditing {len(df)} stocks...")

    # 1. Create a simplified Audit DataFrame
    # We start with just the Ticker
    audit_df = pd.DataFrame()
    audit_df['Ticker'] = df['Ticker']
    audit_df['Sector'] = df['Sector']
    
    # 2. Count Missing Items per stock
    # axis=1 means "operate across the columns for each row"
    audit_df['Missing_Count'] = df.isnull().sum(axis=1)
    
    # 3. Create the "List of Missing Features" column
    # We apply our helper function to every row
    print("Identifying specific missing features (this may take a moment)...")
    audit_df['Missing_Features'] = df.apply(get_missing_columns, axis=1)

    # 4. Sort by "Most Broken" first
    audit_df = audit_df.sort_values(by='Missing_Count', ascending=False)

    # 5. Filter: Only show stocks that are missing *something*
    # (Optional: remove this line if you want to see perfect stocks too)
    audit_df = audit_df[audit_df['Missing_Count'] > 0]

    # 6. Save to CSV
    audit_df.to_csv(OUTPUT_AUDIT_CSV, index=False)
    
    print("-" * 40)
    print(f"DONE. Detailed report saved to: {OUTPUT_AUDIT_CSV}")
    print("-" * 40)
    
    # 7. Print a preview of the worst offenders
    print("\nTOP 5 STOCKS WITH MOST MISSING DATA:")
    print(audit_df[['Ticker', 'Missing_Count']].head(5).to_string(index=False))

    # 8. Interactive Lookup (Optional convenience)
    print("\n--- Interactive Check ---")
    while True:
        target = input("Enter a Ticker to see what it's missing (or 'q' to quit): ").upper()
        if target == 'Q': break
        
        row = audit_df[audit_df['Ticker'] == target]
        if not row.empty:
            print(f"\nMissing Count: {row.iloc[0]['Missing_Count']}")
            print(f"Missing Items: \n{row.iloc[0]['Missing_Features']}")
        else:
            print("Stock has no missing data (or not in DB).")
        print("-" * 20)

if __name__ == "__main__":
    main()

Reading data from market_data_universe.db...
Auditing 18 stocks...
Identifying specific missing features (this may take a moment)...
----------------------------------------
DONE. Detailed report saved to: detailed_missing_data_report.csv
----------------------------------------

TOP 5 STOCKS WITH MOST MISSING DATA:
Ticker  Missing_Count
   ACN              2
   ACN              2
   AAP              1
   AAP              1

--- Interactive Check ---

Missing Count: 1
Missing Items: 
PE_Raw
--------------------

Missing Count: 2
Missing Items: 
EPS_3yr_Growth_Raw, Current_EPS_Change_Raw
--------------------


In [14]:
conn = sqlite3.connect("market_data_universe.db")
df = pd.read_sql("SELECT * FROM fundamentals", conn)
conn.close()

print(f"--- Data Quality Audit ({len(df)} Stocks) ---\n")

# 1. Identify "Ghost" Stocks (Almost no data)
# Count non-null values per row
df['Data_Quality_Score'] = df.notna().sum(axis=1)
ghosts = df[df['Data_Quality_Score'] < 20] # Adjust threshold
if not ghosts.empty:
    print(f"WARNING: {len(ghosts)} stocks have critical data missing:")
    print(ghosts['Ticker'].tolist())
    print("-" * 30)

# 2. Identify "Problematic Columns"
# Show which features are most frequently missing
missing_by_col = df.isnull().mean() * 100
missing_cols = missing_by_col[missing_by_col > 0].sort_values(ascending=False)

print("Top Missing Features (% of stocks):")
print(missing_cols.head(10))

--- Data Quality Audit (18 Stocks) ---

Top Missing Features (% of stocks):
PE_Raw                    11.111111
EPS_3yr_Growth_Raw        11.111111
Current_EPS_Change_Raw    11.111111
dtype: float64


In [15]:
missing_cols

PE_Raw                    11.111111
EPS_3yr_Growth_Raw        11.111111
Current_EPS_Change_Raw    11.111111
dtype: float64

In [130]:
def main():
    conn = sqlite3.connect(DB_NAME)
    
    successful_counts = 0
    
    for ticker in tickers_10:
        print(f"Processing {ticker}...", end=" ")
        
        try:
            stock = yf.Ticker(ticker)
            
            # 1. Get Fundamentals
            fund_data = get_robust_financials(stock)
            if not fund_data: 
                print("Skipping (No Data)")
                continue
            
            # 2. Save to DB
            df = pd.DataFrame([fund_data])
            df['Ticker'] = ticker
            df.to_sql('fundamentals', conn, if_exists='append', index=False)
            
            print("Done.")
            successful_counts += 1
            
            # 3. Sleep to respect API limits
            time.sleep(random.uniform(*DELAY_RANGE))
            
        except Exception as e:
            print(f"Failed: {e}")

    print(f"--- Finished. Successfully collected {successful_counts} stocks. ---")
    print(f"Data saved to {DB_NAME}")
    conn.close()

if __name__ == "__main__":
    main()

Processing MMM... Done.
Processing AOS... Done.
Processing ABT... Done.
Processing ABBV... Done.
Processing ACN... Done.
Processing ATVI... 

$ATVI: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


Skipping (No Data)
Processing AYI... Done.
Processing ADBE... Done.
Processing AAP... Done.
Processing AMD... Done.
--- Finished. Successfully collected 9 stocks. ---
Data saved to sp500_market_data.db


In [132]:
import os # Needed to check if file exists

# --- CONFIGURATION ---
OUTPUT_CSV = "sp500_list1_data.csv"
DELAY_RANGE = (1.0, 2.0) # Sleep seconds

def main():
    print("--- Starting S&P 500 Data Collection (CSV Mode) ---")

    successful_counts = 0
    
    # 2. Check if file exists to handle headers correctly
    # If we are restarting the script, we might want to delete the old file first
    if os.path.exists(OUTPUT_CSV):
        print(f"Note: Appending to existing file {OUTPUT_CSV}")
        # If you want to start fresh every time, uncomment the line below:
        # os.remove(OUTPUT_CSV) 
    
    for ticker in tickers_10:
        print(f"Processing {ticker}...", end=" ")
        
        try:
            stock = yf.Ticker(ticker)
            
            # 3. Get Fundamentals (List 1 Rules)
            feature_data = get_robust_financials(stock)
            
            if not feature_data: 
                print("Skipping (No Data)")
                continue
            
            # 4. Convert to DataFrame (Single Row)
            df_row = pd.DataFrame([feature_data])
            
            # 5. Save to CSV (Append Mode)
            # If file doesn't exist, write header. If it does, skip header.
            file_exists = os.path.isfile(OUTPUT_CSV)
            
            df_row.to_csv(OUTPUT_CSV, mode='a', header=not file_exists, index=False)
            
            print("Saved.")
            successful_counts += 1
            
            # 6. Sleep to respect API limits
            time.sleep(random.uniform(*DELAY_RANGE))
            
        except Exception as e:
            print(f"Failed: {e}")

    print(f"--- Finished. Successfully collected {successful_counts} stocks. ---")
    print(f"Data saved to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

--- Starting S&P 500 Data Collection (CSV Mode) ---
Processing MMM... Saved.
Processing AOS... Saved.
Processing ABT... Saved.
Processing ABBV... Saved.
Processing ACN... Saved.
Processing ATVI... 

$ATVI: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


Skipping (No Data)
Processing AYI... Saved.
Processing ADBE... Saved.
Processing AAP... Saved.
Processing AMD... Saved.
--- Finished. Successfully collected 9 stocks. ---
Data saved to sp500_list1_data.csv


In [1]:
database_stock_trial = pd.read_csv(r'C:\Users\olekb\Projects\sp500_list1_data.csv')

NameError: name 'pd' is not defined

In [134]:
database_stock_trial

,Price_Gt_50MA,50MA_Gt_200MA,RSI_13W_Positive,RSI_13W_Gt_RSI_25W,OBV_20D_Positive,PEG_01_to_05,PEG_Less_1,PEG_Less_1_5,PB_Less_2,Zero_Dividend,MC_to_CF_Less_3,DE_Less_1,Cash_Ratio_Gt_1,Cash_Ratio_Improving,FCF_Positive,FCF_Growing,OCF_Gt_NetIncome,ROA_Positive,Ticker,Sector
0,1,1,1,1,0,0,0,0,0,0,0,0,0,1,1,0,0,1,MMM,Industrials
1,0,1,0,0,0,0,1,1,0,0,0,1,0,0,1,0,1,1,AOS,Industrials
2,0,0,0,0,1,0,1,1,0,0,0,1,0,1,1,1,0,1,ABT,Healthcare
3,1,1,1,1,0,1,1,1,1,0,0,0,0,0,1,0,1,1,ABBV,Healthcare
4,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,ACN,Technology
5,0,1,1,1,0,0,0,0,0,0,0,1,0,0,1,0,1,1,AYI,Industrials
6,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,1,1,1,ADBE,Technology
7,0,1,0,0,0,0,0,1,1,0,0,0,0,1,0,0,1,0,AAP,Consumer Cyclical
8,1,1,1,1,0,0,0,0,0,1,0,1,0,0,1,1,1,1,AMD,Technology


In [68]:
140129580 * 63.61

8913642583.8

In [27]:
def get_itemk(df, keys):
    for k in keys:
        if k in df.index: return df.loc[k]
    return pd.Series([np.nan, np.nan]) 

In [19]:
hist = stock.history(period="2y")

AttributeError: 'list' object has no attribute 'history'

In [21]:
b_s = stock_2.balance_sheet

In [40]:
b_s

,2024-12-31,2023-12-31,2022-12-31,2021-12-31,2020-12-31
Treasury Shares Number,46387979.0,43180265.0,39528515.0,33055027.0,NaN
Ordinary Shares Number,144319615.0,147527327.0,151179079.0,157652567.0,NaN
Share Issued,190577214.0,190577212.0,190577214.0,190577214.0,NaN
Total Debt,216700000.0,155200000.0,366900000.0,219000000.0,NaN
Tangible Book Value,800700000.0,874300000.0,780100000.0,839600000.0,NaN
...,...,...,...,...,...
Allowance For Doubtful Accounts Receivable,-12900000.0,-10100000.0,-9500000.0,-9500000.0,NaN
Gross Accounts Receivable,554300000.0,606100000.0,590700000.0,643900000.0,NaN
Cash Cash Equivalents And Short Term Investments,276100000.0,363400000.0,481800000.0,631400000.0,NaN
Other Short Term Investments,36500000.0,23500000.0,90600000.0,188100000.0,NaN


In [39]:
get_itemk(b_s, ["Total Debt And Capital Lease Obligation", "Total Debt"])

2024-12-31    216700000.0
2023-12-31    155200000.0
2022-12-31    366900000.0
2021-12-31    219000000.0
2020-12-31            NaN
Name: Total Debt, dtype: float64

In [24]:
b_s.index

Index(['Treasury Shares Number', 'Ordinary Shares Number', 'Share Issued',
       'Total Debt', 'Tangible Book Value', 'Invested Capital',
       'Working Capital', 'Net Tangible Assets', 'Capital Lease Obligations',
       'Common Stock Equity', 'Total Capitalization',
       'Total Equity Gross Minority Interest', 'Stockholders Equity',
       'Gains Losses Not Affecting Retained Earnings',
       'Other Equity Adjustments', 'Foreign Currency Translation Adjustments',
       'Minimum Pension Liabilities', 'Treasury Stock', 'Retained Earnings',
       'Additional Paid In Capital', 'Capital Stock', 'Common Stock',
       'Preferred Stock', 'Total Liabilities Net Minority Interest',
       'Total Non Current Liabilities Net Minority Interest',
       'Other Non Current Liabilities', 'Employee Benefits',
       'Non Current Pension And Other Postretirement Benefit Plans',
       'Non Current Accrued Expenses',
       'Long Term Debt And Capital Lease Obligation',
       'Long Term Capita

In [52]:
stock_2 = yf.Ticker('ACN')

In [ ]:
EPS_3yr_Growth_Raw

In [53]:
financial = stock_2.financials
b_s_2 = stock_2.balance_sheet
b_s_2_info = stock_2.info
c_f = stock_2.cashflow

In [56]:
b_s_2_info

{'address1': '1 Grand Canal Square',
 'address2': 'Grand Canal Harbour',
 'city': 'Dublin',
 'zip': 'D02 P820',
 'country': 'Ireland',
 'phone': '353 1 646 2000',
 'fax': '353 1 646 2020',
 'website': 'https://www.accenture.com',
 'industry': 'Information Technology Services',
 'industryKey': 'information-technology-services',
 'industryDisp': 'Information Technology Services',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Accenture plc provides strategy and consulting, industry X, song, and technology and operation services in the Americas, Europe, the Middle East, Africa, and the Asia Pacific. It offers systems integration and application management; security; intelligent platform; infrastructure; software engineering; data, AI, cloud; and automation and global delivery services. The company also operates business processes for specific enterprise functions, including finance and accounting, sourcing and procurement, supply

In [ ]:
AAP PE_Raw, EPS_3yr_Growth_Raw

In [23]:
b_s_2_info['returnOnEquity']

0.28209

In [ ]:
0.28209

In [24]:
roe_calc = 5.336000e+08 / 1.883500e+09
print(roe_calc)

0.28330236262277675


In [ ]:
n_income = 5.336000e+08
equity =  1.883500e+09

In [108]:
c_f.loc['Free Cash Flow']

2024-12-31    473800000.0
2023-12-31    597700000.0
2022-12-31    321100000.0
2021-12-31    566000000.0
Name: Free Cash Flow, dtype: float64

In [101]:
b_s.loc['Capital Lease Obligations']

2024-12-31    23500000.0
2023-12-31    27900000.0
2022-12-31    22400000.0
2021-12-31    22300000.0
2020-12-31           NaN
Name: Capital Lease Obligations, dtype: float64

In [ ]:
free cash flow
473800000.0

In [ ]:
operating cash flow 
581800000.0	

In [ ]:
capex
-108000000.0

In [109]:
581800000.0	- 108000000.0

473800000.0

In [70]:
cazh

,2024-12-31,2023-12-31,2022-12-31,2021-12-31
Free Cash Flow,473800000.0,597700000.0,321100000.0,566000000.0
Repurchase Of Capital Stock,-305800000.0,-306500000.0,-403500000.0,-366500000.0
Capital Expenditure,-108000000.0,-72600000.0,-70300000.0,-75100000.0
End Cash Position,239600000.0,339900000.0,391200000.0,443300000.0
Beginning Cash Position,339900000.0,391200000.0,443300000.0,573100000.0
Effect Of Exchange Rate Changes,-6600000.0,-12800000.0,-20800000.0,0.0
Changes In Cash,-93700000.0,-38500000.0,-31300000.0,-129800000.0
Financing Cash Flow,-408400000.0,-684700000.0,-430800000.0,-421000000.0
Cash Flow From Continuing Financing Activities,-408400000.0,-684700000.0,-430800000.0,-421000000.0
Net Other Financing Charges,NaN,NaN,-700000.0,NaN


In [74]:
cf = get_itemk(cazh, ['Operating Cash Flow', 'Cash Flow From Continuing Operating Activities'])

In [76]:
gink = 1 if (pd.notna(cf) and cf > 0 and (8913642583.8/cf) < 3) else 0

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [75]:
cf

2024-12-31    581800000.0
2023-12-31    670300000.0
2022-12-31    391400000.0
2021-12-31    641100000.0
Name: Operating Cash Flow, dtype: float64

In [51]:
hist_weeklyz = hist['Close'].resample('W-FRI').last()

In [52]:
deltaz = hist_weeklyz.diff()

In [54]:
hist_weeklyz

Date
2023-11-24 00:00:00-05:00     76.101311
2023-12-01 00:00:00-05:00     79.194542
2023-12-08 00:00:00-05:00     81.986359
2023-12-15 00:00:00-05:00     84.849594
2023-12-22 00:00:00-05:00     84.334053
                                ...    
2025-10-24 00:00:00-04:00    167.779068
2025-10-31 00:00:00-04:00    165.787628
2025-11-07 00:00:00-05:00    164.134720
2025-11-14 00:00:00-05:00    167.580002
2025-11-21 00:00:00-05:00    166.580002
Freq: W-FRI, Name: Close, Length: 105, dtype: float64

In [53]:
deltaz

Date
2023-11-24 00:00:00-05:00          NaN
2023-12-01 00:00:00-05:00     3.093231
2023-12-08 00:00:00-05:00     2.791817
2023-12-15 00:00:00-05:00     2.863235
2023-12-22 00:00:00-05:00    -0.515541
                               ...    
2025-10-24 00:00:00-04:00    15.792145
2025-10-31 00:00:00-04:00    -1.991440
2025-11-07 00:00:00-05:00    -1.652908
2025-11-14 00:00:00-05:00     3.445282
2025-11-21 00:00:00-05:00    -1.000000
Freq: W-FRI, Name: Close, Length: 105, dtype: float64

In [14]:
infoz = stock.info

In [21]:
infoz

{'address1': '3M Center',
 'city': 'Saint Paul',
 'state': 'MN',
 'zip': '55144-1000',
 'country': 'United States',
 'phone': '651 733 1110',
 'website': 'https://www.3m.com',
 'industry': 'Conglomerates',
 'industryKey': 'conglomerates',
 'industryDisp': 'Conglomerates',
 'sector': 'Industrials',
 'sectorKey': 'industrials',
 'sectorDisp': 'Industrials',
 'longBusinessSummary': '3M Company provides diversified technology services in the Americas, the Asia Pacific, Europe, the Middle East, Africa, and internationally. It operates through three segments: Safety and Industrial, Transportation and Electronics, and Consumer. The Safety and Industrial segment offers industrial abrasives and finishing for metalworking applications; autobody repair solutions; industrial specialty products, such as personal hygiene products, masking, and packaging materials; electrical products and materials for construction and maintenance, power distribution, and electrical original equipment manufacturers; 

In [3]:
stock_2 = yf.Ticker(tickers[1])

In [118]:
stock

yfinance.Ticker object <MMM>

In [ ]:
stock_2.financials

AttributeError: 'Ticker' object has no attribute 'financial'

In [122]:
financials = get_robust_financials(stock)

In [123]:
financials

{'Price_Gt_50MA': 1,
 '50MA_Gt_200MA': 1,
 'RSI_13W_Positive': 1,
 'RSI_13W_Gt_RSI_25W': 1,
 'OBV_20D_Positive': 0,
 'PEG_01_to_05': 0,
 'PEG_Less_1': 0,
 'PEG_Less_1_5': 0,
 'PB_Less_2': 0,
 'Zero_Dividend': 0,
 'MC_to_CF_Less_3': 0,
 'DE_Less_1': 0,
 'Cash_Ratio_Gt_1': 0,
 'Cash_Ratio_Improving': 1,
 'FCF_Positive': 1,
 'FCF_Growing': 0,
 'OCF_Gt_NetIncome': 0,
 'ROA_Positive': 1,
 'Ticker': 'MMM',
 'Sector': 'Industrials'}

In [ ]:
from config import REPORTING_LAG_DAYS
from datetime import timedelta

In [ ]:
def calculate_rsi(series, period=14):
    '''Calculate RSI using Wilder's smoothing method.'''
    
    delta = series.diff()
    # Gains and Losses 
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    # Wilder's Exponential Weighted Mean (1/period = alpha)
    
    # Subsequent averages: exponential smoothing
    # Wilder's smoothing: alpha = 1/period
    # adjust = False to mathc the formula 
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    
    # RS
    rs = avg_gain / avg_loss
    
    # RSI
    return 100 - (100 / (1 + rs))

def calculate_obv(df):
    '''Function to calculate on Balance Volume (OBV)'''
    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
    return df['OBV']

def get_robust_financials(ticker_symbol, hist, fin, bs, cash, info, screening_date):
    features = {}
    
    try:
        # 1. EARLY EXIT IF DATA IS EMPTY
        if hist.empty or fin.empty or bs.empty: 
            return None
        
        # --- UNIVERSAL HELPER: Get Single Value Safely ---
        # Replaces both get_item and old get_val
        def get_val(df, keys, iloc_idx=0):
            for k in keys:
                if k in df.index:
                    try:
                        val = df.loc[k].iloc[iloc_idx]
                        return val
                    except IndexError:
                        return np.nan
            return np.nan

        # --- 1a. TECHNICAL INDICATORS ---
        current_price = hist['Close'].iloc[-1]
        
        ma_50 = hist['Close'].rolling(window=50).mean().iloc[-1]
        ma_200 = hist['Close'].rolling(window=200).mean().iloc[-1]
        
        # RSI (Weekly)
        hist_weekly = hist['Close'].resample('W-FRI').last()
        rsi_13_wk = calculate_rsi(hist_weekly, period=13).iloc[-1]
        rsi_25_wk = calculate_rsi(hist_weekly, period=25).iloc[-1]
        
        # OBV
        obv_series = calculate_obv(hist)
        obv_20d_change = obv_series.iloc[-1] - obv_series.iloc[-21] if len(obv_series) > 20 else 0
        
        features['Price_Gt_50MA'] = 1 if current_price > ma_50 else 0
        features['50MA_Gt_200MA'] = 1 if ma_50 > ma_200 else 0
        features['RSI_13W_Gt_RSI_25W'] = 1 if rsi_13_wk > rsi_25_wk else 0
        features['OBV_20D_Positive'] = 1 if obv_20d_change > 0 else 0

        # --- 1b. VALUATION & RATIOS (Calculated) ---
        
        # EPS Data (Using get_val avoids the IndexError crash)
        eps_curr = get_val(fin, ["Basic EPS"], iloc_idx=0)
        eps_prev = get_val(fin, ["Basic EPS"], iloc_idx=1)
        eps_prev_3 = get_val(fin, ["Basic EPS"], iloc_idx=3) # Safely returns NaN if not found
        
        trailing_pe = np.nan
        if pd.notna(current_price) and pd.notna(eps_curr) and eps_curr > 0:
            trailing_pe = current_price / eps_curr
            
        # Fallback (calculate manually
        if pd.isna(trailing_pe):
            # calculate only if EPS positive
            if pd.notna(current_price) and pd.notna(eps_curr) and eps_curr > 0:
                trailing_pe = current_price / eps_curr
            else:
                # If EPS is negative or missing, P/E is undefined (NaN).
                # We do NOT want a negative number here, as it would mess up ranking.
                trailing_pe = np.nan
        
        # Growth Rates
        eps_growth_3yr = np.nan
        
        if pd.notna(eps_curr) and pd.notna(eps_prev_3) and abs(eps_prev_3) > 0:
            
            # CASE A: Standard CAGR (Both Positive)
            # This is the most accurate financial metric for stable companies
            if eps_curr > 0 and eps_prev_3 > 0:
                eps_growth_3yr = (eps_curr / eps_prev_3)**(1/3) - 1
                
            # CASE B: Negative/Mixed Values (Linear Approximation)
            # CAGR formula fails here, so we use: (Total % Change) / 3 Years
            # Formula: ((End - Start) / |Start|) / 3
            else:
                total_change_pct = (eps_curr - eps_prev_3) / abs(eps_prev_3)
                eps_growth_3yr = total_change_pct / 3

        # PEG
        manual_peg = np.nan
        if pd.notna(trailing_pe) and pd.notna(eps_growth_3yr) and eps_growth_3yr > 0:
            manual_peg = trailing_pe / (eps_growth_3yr * 100)
        
        final_peg = manual_peg if pd.notna(manual_peg) else info.get('trailingPegRatio')
        
        features['PEG_01_to_05'] = 1 if (final_peg and 0.1 < final_peg <= 0.5) else 0
        features['PEG_Less_1'] = 1 if (final_peg and final_peg < 1) else 0
        features['PEG_Less_1_5'] = 1 if (final_peg and final_peg < 1.5) else 0 
        
        # Other List 1 Ratios
        
        # market cap
        shares_out = get_val(bs, ['Ordinary Shares Number', 'Share Issued', 'Basic Average Shares'], iloc_idx=0)
        mkt_cap = np.nan
        if pd.notna(current_price) and pd.notna(shares_out) and shares_out > 0:
            mkt_cap = current_price * shares_out
            
        # price to book
        pb = np.nan
        equity_curr = get_val(bs, ["Total Stockholder Equity", "Stockholders Equity"], iloc_idx=0)
        if pd.notna(mkt_cap) and pd.notna(equity_curr) and equity_curr > 0:
            pb = mkt_cap / equity_curr
        
        # dividend yield
        div_yield = np.nan
        div_paid = get_val(cash, ['Cash Dividends Paid', 'Dividends Paid', 'Cash Dividend Paid'], iloc_idx=0)
        if pd.notna(div_paid) and pd.notna(mkt_cap) and mkt_cap > 0:
            div_yield = abs(div_paid) / mkt_cap
        
        ocf_curr = get_val(cash, ["Operating Cash Flow", "Total Cash From Operating Activities"], iloc_idx=0)
        
        if pd.notna(mkt_cap) and pd.notna(ocf_curr) and ocf_curr > 0:
            features['MC_to_CF_Less_3'] = 1 if (mkt_cap / ocf_curr) < 3 else 0
        else:
            features['MC_to_CF_Less_3'] = 0

        # --- 1c. FINANCIAL HEALTH ---
        
        # Fetch Balance Sheet Items
        debt_curr = get_val(bs, ["Total Debt And Capital Lease Obligation", "Total Debt"], iloc_idx=0)
        equity_curr = get_val(bs, ["Total Stockholder Equity", "Stockholders Equity"], iloc_idx=0)
        equity_prev = get_val(bs, ["Total Stockholder Equity", "Stockholders Equity"], iloc_idx=1)
        equity_prev_2 = get_val(bs, ["Total Stockholder Equity", "Stockholders Equity"], iloc_idx=2)
        
        if pd.notna(debt_curr) and pd.notna(equity_curr) and equity_curr != 0:
            features['DE_Less_1'] = 1 if (debt_curr / equity_curr) < 1 else 0
        else:
            features['DE_Less_1'] = 0
            
        # Cash Ratios
        cash_curr = get_val(bs, ["Cash And Cash Equivalents", "Cash Financial"], iloc_idx=0)
        cash_prev = get_val(bs, ["Cash And Cash Equivalents", "Cash Financial"], iloc_idx=1)
        cliab_curr = get_val(bs, ["Current Liabilities", "Total Current Liabilities"], iloc_idx=0)
        cliab_prev = get_val(bs, ["Current Liabilities", "Total Current Liabilities"], iloc_idx=1)
        
        if pd.notna(cash_curr) and pd.notna(cliab_curr) and cliab_curr != 0:
            cash_ratio = cash_curr / cliab_curr
            features['Cash_Ratio_Gt_1'] = 1 if cash_ratio > 1 else 0
            
            if pd.notna(cash_prev) and pd.notna(cliab_prev) and cliab_prev != 0:
                features['Cash_Ratio_Improving'] = 1 if cash_ratio > (cash_prev / cliab_prev) else 0
            else:
                features['Cash_Ratio_Improving'] = 0
        else:
            features['Cash_Ratio_Gt_1'] = 0
            features['Cash_Ratio_Improving'] = 0

        # FCF
        capex_curr = get_val(cash, ["Capital Expenditure", "Capital Expenditures", "Capital Expenditure Reported"], iloc_idx=0)
        capex_prev = get_val(cash, ["Capital Expenditure", "Capital Expenditures", "Capital Expenditure Reported"], iloc_idx=1)
        capex_prev_2 = get_val(cash, ["Capital Expenditure", "Capital Expenditures", "Capital Expenditure Reported"], iloc_idx=2)
        ocf_prev = get_val(cash, ["Operating Cash Flow", "Total Cash From Operating Activities"], iloc_idx=1)
        ocf_prev_2 = get_val(cash, ["Operating Cash Flow", "Total Cash From Operating Activities"], iloc_idx=2)
        
        fcf_curr = (ocf_curr + capex_curr) if (pd.notna(ocf_curr) and pd.notna(capex_curr)) else np.nan
        fcf_prev = (ocf_prev + capex_prev) if (pd.notna(ocf_prev) and pd.notna(capex_prev)) else np.nan
        fcf_prev_2 = (ocf_prev_2 + capex_prev_2) if (pd.notna(ocf_prev_2) and pd.notna(capex_prev_2)) else np.nan
        
        features['FCF_Positive'] = 1 if (pd.notna(fcf_curr) and fcf_curr > 0) else 0
        features['FCF_Growing'] = 1 if (pd.notna(fcf_curr) and pd.notna(fcf_prev) and fcf_curr > fcf_prev) else 0
        
        if pd.notna(fcf_curr) and pd.notna(fcf_prev) and pd.notna(fcf_prev_2):
            fcf_streak = (fcf_curr > fcf_prev) and (fcf_prev > fcf_prev_2)
            features['FCF_Growing_3Yrs_Streak'] = 1 if fcf_streak else 0
        else:
            features['FCF_Growing_3Yrs_Streak'] = 0
        
        # OCF > Net Income
        ni_curr = get_val(fin, ["Net Income", "Net Income Common Stockholders"], iloc_idx=0)
        ni_prev = get_val(fin, ["Net Income", "Net Income Common Stockholders"], iloc_idx=1) 
        ni_prev_2 = get_val(fin, ["Net Income", "Net Income Common Stockholders"], iloc_idx=2)
        ni_prev_3 = get_val(fin, ["Net Income", "Net Income Common Stockholders"], iloc_idx=3)
        features['OCF_Gt_NetIncome'] = 1 if (pd.notna(ocf_curr) and pd.notna(ni_curr) and ocf_curr > ni_curr) else 0
        
        # ROA Positive
        assets_curr = get_val(bs, ["Total Assets"], iloc_idx=0)
        features['ROA_Positive'] = 1 if (pd.notna(ni_curr) and pd.notna(assets_curr) and assets_curr > 0 and ni_curr > 0) else 0
        
        # Fetch Raw Data
        rev_curr = get_val(fin, ['Total Revenue', 'Operating Revenue'], iloc_idx=0)
        rev_prev = get_val(fin, ['Total Revenue', 'Operating Revenue'], iloc_idx=1)
        rev_prev_2 = get_val(fin, ['Total Revenue', 'Operating Revenue'], iloc_idx=2)
        rev_prev_3 = get_val(fin, ['Total Revenue', 'Operating Revenue'], iloc_idx=3)    
        
        # 1d. List 3: Historical Data
        roe_curr = (ni_curr / equity_curr) if (pd.notna(ni_curr) and pd.notna(equity_curr) and equity_curr > 0) else 0
        roe_prev = (ni_prev / equity_prev) if (pd.notna(ni_prev) and pd.notna(equity_prev) and equity_prev > 0) else 0
        roe_prev_2 = (ni_prev_2 / equity_prev_2) if (pd.notna(ni_prev_2) and pd.notna(equity_prev_2) and equity_prev_2 > 0) else 0
        
        features['ROE_Gt_15_3Yrs'] = 1 if (roe_curr > 0.15 and roe_prev > 0.15 and roe_prev_2 > 0.15) else 0
        if pd.notna(ni_curr) and pd.notna(ni_prev_3) and ni_prev_3 > 0 and ni_curr > 0:
            ni_cagr = (ni_curr / ni_prev_3)**(1/3) - 1
            features['Net_Income_Growth_Gt_8pct'] = 1 if ni_cagr > 0.08 else 0
        else:
            features['Net_Income_Growth_Gt_8pct'] = 0  
            
        # Calculate R&D Growth (3 Year CAGR) multiple scenarios to ensure recent R&D spends are considered
        rnd_curr = get_val(fin, ["Research And Development"], iloc_idx=0)
        rnd_prev = get_val(fin, ["Research And Development"], iloc_idx=1)
        rnd_prev_2 = get_val(fin, ["Research And Development"], iloc_idx=2)
        rnd_prev_3 = get_val(fin, ["Research And Development"], iloc_idx=3)
        
        sales_vs_rnd_flag = 0 # Default to 0
        
        # Case 1: Try 3-Year CAGR for BOTH
        if (pd.notna(rev_curr) and pd.notna(rev_prev_3) and rev_prev_3 > 0 and 
            pd.notna(rnd_curr) and pd.notna(rnd_prev_3) and rnd_prev_3 > 0):
            
            s_cagr = (rev_curr / rev_prev_3)**(1/3) - 1
            r_cagr = (rnd_curr / rnd_prev_3)**(1/3) - 1
            sales_vs_rnd_flag = 1 if s_cagr > r_cagr else 0

        # Case 2: Try 2-Year CAGR for BOTH
        elif (pd.notna(rev_curr) and pd.notna(rev_prev_2) and rev_prev_2 > 0 and 
              pd.notna(rnd_curr) and pd.notna(rnd_prev_2) and rnd_prev_2 > 0):
            
            s_cagr = (rev_curr / rev_prev_2)**(1/2) - 1
            r_cagr = (rnd_curr / rnd_prev_2)**(1/2) - 1
            sales_vs_rnd_flag = 1 if s_cagr > r_cagr else 0
            
        # Case 3: Try 1-Year Growth for BOTH
        elif (pd.notna(rev_curr) and pd.notna(rev_prev) and rev_prev > 0 and 
              pd.notna(rnd_curr) and pd.notna(rnd_prev) and rnd_prev > 0):
            
            s_cagr = (rev_curr / rev_prev) - 1
            r_cagr = (rnd_curr / rnd_prev) - 1
            sales_vs_rnd_flag = 1 if s_cagr > r_cagr else 0
            
        # Case 4: Company has Sales Growth but no R&D 
        elif pd.isna(rnd_curr) or rnd_curr == 0:
             # just check for growth
             if pd.notna(rev_curr) and pd.notna(rev_prev) and rev_curr > rev_prev:
                 sales_vs_rnd_flag = 1

        features['Sales_Growth_Gt_RnD_Growth'] = sales_vs_rnd_flag
        
        op_inc_curr = get_val(fin, ['Operating Income'], iloc_idx=0)
        op_inc_prev = get_val(fin, ['Operating Income'], iloc_idx=1)
        op_inc_prev_2 = get_val(fin, ['Operating Income'], iloc_idx=2)
        
        # margins
        om_curr = (op_inc_curr / rev_curr) if (pd.notna(op_inc_curr) and pd.notna(rev_curr)) else 0
        om_prev = (op_inc_prev / rev_prev) if (pd.notna(op_inc_prev) and pd.notna(rev_prev)) else 0
        om_prev_2 = (op_inc_prev_2 / rev_prev_2) if (pd.notna(op_inc_prev_2) and pd.notna(rev_prev_2)) else 0
        
        # average (3 years)
        om_avg_past = (om_curr + om_prev + om_prev_2) / 3
        
        features['Op_Margin_Gt_3yr_Avg'] = 1 if om_curr > om_avg_past else 0
        
        # --- 2. METADATA (Raw Data for List 2) ---
        
        # A. Fetch Raw Data (the ones that are missing)
        
        gross_profit_curr = get_val(fin, ['Gross Profit'], iloc_idx=0)
        
        total_liab_curr = get_val(bs, ['Total Liabilities Net Minority Interest', 'Total Liabilities'], iloc_idx=0)
        if pd.isna(total_liab_curr):
             total_liab_curr = (assets_curr - equity_curr) if (pd.notna(assets_curr) and pd.notna(equity_curr)) else np.nan

        # B. Store Metadata
        features['Ticker'] = ticker_symbol
        features['Sector'] = info.get('sector', 'Unknown')
        features['Market_Cap_Raw'] = mkt_cap 
        features['PE_Raw'] = trailing_pe
        features['Free_Cash_Flow_Raw'] = fcf_curr
        features['PB_Raw'] = pb
        features['Div_Yield_Raw'] = div_yield
        features['ROE_Raw'] = info.get('returnOnEquity')

        # C. Calculate Raw Ratios
        
        # Net Profit Margins (Current, Prev, Prev-2)
        if pd.notna(ni_curr) and pd.notna(rev_curr) and rev_curr > 0:
            features['Net_Profit_Margin_Raw'] = ni_curr / rev_curr
        else:
            features['Net_Profit_Margin_Raw'] = np.nan

        if pd.notna(ni_prev) and pd.notna(rev_prev) and rev_prev > 0:
            features['Net_Profit_Margin_Prev_Raw'] = ni_prev / rev_prev
        else:
            features['Net_Profit_Margin_Prev_Raw'] = np.nan

        if pd.notna(ni_prev_2) and pd.notna(rev_prev_2) and rev_prev_2 > 0:
            features['Net_Profit_Margin_Prev_2_Raw'] = ni_prev_2 / rev_prev_2
        else:
            features['Net_Profit_Margin_Prev_2_Raw'] = np.nan

        # Gross & Operating Margins
        if pd.notna(gross_profit_curr) and pd.notna(rev_curr) and rev_curr > 0:
            features['Gross_Profit_Margin_Raw'] = gross_profit_curr / rev_curr
        else:
            features['Gross_Profit_Margin_Raw'] = np.nan

        if pd.notna(op_inc_curr) and pd.notna(rev_curr) and rev_curr > 0:
            features['Operating_Margin_Raw'] = op_inc_curr / rev_curr
        else:
            features['Operating_Margin_Raw'] = np.nan

        # Sales Growth (3yr)
        if pd.notna(rev_curr) and pd.notna(rev_prev_3) and rev_prev_3 > 0:
             features['Sales_3yr_Growth_Raw'] = (rev_curr / rev_prev_3)**(1/3) - 1
        else:
             features['Sales_3yr_Growth_Raw'] = np.nan

        # Debt & Solvency Ratios
        if pd.notna(total_liab_curr) and pd.notna(assets_curr) and assets_curr > 0:
            features['Debt_To_Assets_Ratio_Raw'] = total_liab_curr / assets_curr
            features['Assets_To_Liability_Ratio_Raw'] = assets_curr / total_liab_curr if total_liab_curr > 0 else np.nan
        else:
            features['Debt_To_Assets_Ratio_Raw'] = np.nan
            features['Assets_To_Liability_Ratio_Raw'] = np.nan

        if pd.notna(debt_curr) and pd.notna(equity_curr) and equity_curr != 0:
            features['Long_Term_Debt_To_Equity_Raw'] = debt_curr / equity_curr
        else:
            features['Long_Term_Debt_To_Equity_Raw'] = np.nan

        if pd.notna(rev_curr) and pd.notna(assets_curr) and assets_curr > 0:
            features['Sales_To_Assets_Ratio_Raw'] = rev_curr / assets_curr
        else:
            features['Sales_To_Assets_Ratio_Raw'] = np.nan

        # EPS Growth Stats
        features['EPS_3yr_Growth_Raw'] = eps_growth_3yr
        
        if pd.notna(eps_curr) and pd.notna(eps_prev) and abs(eps_prev) > 0:
             features['Current_EPS_Change_Raw'] = (eps_curr - eps_prev) / abs(eps_prev)
        else:
             features['Current_EPS_Change_Raw'] = np.nan
             
        features['Screening_Date'] = screening_date
        features['Sector'] = info.get('sector', 'Unknown')

        return features
    
    except Exception as e:
        # print(f"Error on {ticker_symbol}: {e}")
        return None

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta

REPORTING_LAG_DAYS = 90 # Assume it takes 90 days for Q4 earnings to become public

def process_historical_snapshot(ticker, full_hist, full_fin, full_bs, full_cash, info, screening_date_str):
    """
    Slices data to simulate what was known on the screening_date, then calculates features.
    """
    screening_date = pd.to_datetime(screening_date_str)
    
    # 1. SLICE PRICE HISTORY
    # Keep only prices on or before the screening date
    # (Assuming full_hist index is a DatetimeIndex)
    hist_sliced = full_hist[full_hist.index <= screening_date].copy()
    
    # Need at least a few months of data for 200MA
    if len(hist_sliced) < 200:
        return None 

    # 2. FILTER FINANCIAL STATEMENTS (The "Lag" Rule)
    # You can't use a financial statement filed on Dec 31 if today is Jan 1. 
    # You must wait for the report to be published (~90 days later).
    cutoff_date = screening_date - timedelta(days=REPORTING_LAG_DAYS)
    
    # Ensure statement columns are datetime objects, then filter
    def filter_statement(df):
        if df is None or df.empty: return df
        # Keep columns where the date is older than the cutoff
        valid_cols = [col for col in df.columns if pd.to_datetime(col) <= cutoff_date]
        return df[valid_cols]

    fin_filtered = filter_statement(full_fin)
    bs_filtered = filter_statement(full_bs)
    cash_filtered = filter_statement(full_cash)

    # 3. PASS TO YOUR CALCULATOR
    # Because we filtered the columns newest-first, iloc[0] inside this function 
    # will naturally be the correct "Current Year" for that specific screening date.
    return get_robust_financials(
        ticker_symbol=ticker, 
        hist=hist_sliced, 
        fin=fin_filtered, 
        bs=bs_filtered, 
        cash=cash_filtered, 
        info=info, 
        screening_date=screening_date_str
    )

def run_backtest_pipeline(tickers, screening_dates):
    """
    Main loop to generate multiple historical snapshots for your dataset.
    """
    all_snapshots = []
    
    for ticker in tickers:
        print(f"Fetching raw data for {ticker}...")
        stock = yf.Ticker(ticker)
        
        # Pull the MAXIMUM available data once per ticker
        try:
            full_hist = stock.history(period="max")
            full_fin = stock.financials
            full_bs = stock.balance_sheet
            full_cash = stock.cashflow
            info = stock.info
        except Exception as e:
            print(f"Failed to fetch {ticker}: {e}")
            continue
            
        # Generate a row for each date we want to test
        for date_str in screening_dates:
            row_data = process_historical_snapshot(
                ticker, full_hist, full_fin, full_bs, full_cash, info, date_str
            )
            
            if row_data:
                all_snapshots.append(row_data)
                
    # Convert all gathered rows into a DataFrame and save
    final_df = pd.DataFrame(all_snapshots)
    return final_df

# Example Usage:
# dates_to_test = ["2022-06-30", "2022-12-31", "2023-06-30", "2023-12-31"]
# dataset = run_backtest_pipeline(["AAPL", "MSFT"], dates_to_test)
# dataset.to_sql('historical_fundamentals', conn, if_exists='append', index=False)

In [3]:
def check_yfinance_depth(ticker_symbol):
    print(f"\n🔍 Probing Data Depth for: {ticker_symbol}")
    print("-" * 40)
    
    stock = yf.Ticker(ticker_symbol)
    
    # --- 1. PRICE HISTORY ---
    try:
        hist = stock.history(period="max")
        if not hist.empty:
            earliest_price = hist.index.min().strftime('%Y-%m-%d')
            latest_price = hist.index.max().strftime('%Y-%m-%d')
            print(f"📈 Price History:   {earliest_price}  to  {latest_price} ({len(hist)} days)")
        else:
            print("📈 Price History:   [No Data]")
    except Exception as e:
        print(f"📈 Price History:   [Error: {e}]")

    # --- HELPER FOR FINANCIALS ---
    # yfinance puts dates as the columns, ordered newest to oldest
    def get_statement_depth(df, name):
        if df is not None and not df.empty:
            # Get the oldest (min) and newest (max) column dates
            earliest = df.columns.min().strftime('%Y-%m-%d')
            latest = df.columns.max().strftime('%Y-%m-%d')
            num_periods = len(df.columns)
            print(f"📊 {name:<17} {earliest}  to  {latest} ({num_periods} periods)")
        else:
            print(f"📊 {name:<17} [No Data]")

    # --- 2. ANNUAL STATEMENTS ---
    print("\n--- Annual Accounting ---")
    get_statement_depth(stock.financials, "Income Statement:")
    get_statement_depth(stock.balance_sheet, "Balance Sheet:")
    get_statement_depth(stock.cashflow, "Cash Flow:")

    # --- 3. QUARTERLY STATEMENTS ---
    print("\n--- Quarterly Accounting ---")
    get_statement_depth(stock.quarterly_financials, "Income Statement:")
    get_statement_depth(stock.quarterly_balance_sheet, "Balance Sheet:")
    get_statement_depth(stock.quarterly_cashflow, "Cash Flow:")
    
    print("-" * 40)

# Run the test on a mature company (AAPL) and a newer one (ABNB)
if __name__ == "__main__":
    check_yfinance_depth("AAPL")
    check_yfinance_depth("ABNB")


🔍 Probing Data Depth for: AAPL
----------------------------------------
📈 Price History:   1980-12-12  to  2026-03-24 (11411 days)

--- Annual Accounting ---
📊 Income Statement: 2021-09-30  to  2025-09-30 (5 periods)
📊 Balance Sheet:    2021-09-30  to  2025-09-30 (5 periods)
📊 Cash Flow:        2021-09-30  to  2025-09-30 (5 periods)

--- Quarterly Accounting ---
📊 Income Statement: 2024-12-31  to  2025-12-31 (5 periods)
📊 Balance Sheet:    2024-09-30  to  2025-12-31 (6 periods)
📊 Cash Flow:        2024-06-30  to  2025-12-31 (7 periods)
----------------------------------------

🔍 Probing Data Depth for: ABNB
----------------------------------------
📈 Price History:   2020-12-10  to  2026-03-24 (1326 days)

--- Annual Accounting ---
📊 Income Statement: 2021-12-31  to  2025-12-31 (5 periods)
📊 Balance Sheet:    2021-12-31  to  2025-12-31 (5 periods)
📊 Cash Flow:        2021-12-31  to  2025-12-31 (5 periods)

--- Quarterly Accounting ---
📊 Income Statement: 2024-06-30  to  2025-12-31 (7 p